In [1]:
import pandas as pd
df = pd.read_csv("data/egypt_real_estate_listings.csv")
df.columns.tolist()


['url',
 'price',
 'description',
 'location',
 'type',
 'size',
 'bedrooms',
 'bathrooms',
 'available_from',
 'payment_method',
 'down_payment']

In [2]:
df["description"].head(50)

0     OWN A CHALET IN EL GOUNA WITH A PRIME LOCATION...
1     For sale, a villa with immediate delivery in C...
2     With a down payment of EGP 1,513,000, a fully ...
3     Own an apartment in New Cairo with a minimal d...
4     Project: Granville\nLocation: Fifth Settlement...
5     Chalet with Marina and Lake View in The Island...
6     Penthouse 4BR for sale with Installments over ...
7     Stand-alone villa in NYOUM Compound, October\n...
8     **La Vista Ras El Hikma seaview twinhouse - Re...
9     A 195-square-meter duplex + a 42-square-meter ...
10    With Down payment 1 million only Fully Finishe...
11    • Chalet In || Gouna ||:\n- Layout: 1 Bedroom,...
12    Fully finished chalet with air conditioning fo...
13    Luxury Chalet 3 Bedroom With Full Sea View In ...
14    Looking for elegant apartment with luxurious f...
15    Unit Type: Duplex\n2 Bedrooms\nBuilt Up Area: ...
16    Taj City Compound - New Cairo\nFifth Settlemen...
17    Developer: Orascom\nProject: Makadi Height

In [3]:
df["description"].str.contains("[\u0600-\u06FF]").sum()


1783

In [4]:
english = df[~df["description"].str.contains("[\u0600-\u06FF]", na=False)]



In [5]:
arabic = df[df["description"].str.contains("[\u0600-\u06FF]", na=False)]

In [6]:
en_sample = english.sample(50, random_state=42)
ar_sample = arabic.sample(50, random_state=42)


In [7]:
sample = pd.concat([en_sample, ar_sample])


In [8]:
import tiktoken
gpt4o_enc = tiktoken.encoding_for_model("gpt-4o")


In [9]:
cl100k_enc = tiktoken.get_encoding("cl100k_base")


KeyboardInterrupt: 

In [ ]:
from transformers import AutoTokenizer
bert_tok = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")


In [ ]:
    ara_tok = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv2")


In [ ]:
sample["gpt4o_tokens"] = sample["description"].apply(lambda x: len(gpt4o_enc.encode(x)))


In [ ]:
sample["cl100k_tokens"] = sample["description"].apply(lambda x: len(cl100k_enc.encode(x)))
sample["bert_tokens"] = sample["description"].apply(lambda x: len(bert_tok.encode(x)))
sample["ara_tokens"] = sample["description"].apply(lambda x: len(ara_tok.encode(x)))


In [ ]:
sample[["description", "gpt4o_tokens", "cl100k_tokens", "bert_tokens", "ara_tokens"]].head(10)

In [ ]:
sample[["description", "gpt4o_tokens", "cl100k_tokens", "bert_tokens", "ara_tokens"]].tail(10)

In [ ]:
sample["lang"] = ["en"]*50 + ["ar"]*50
sample.groupby("lang")[["gpt4o_tokens", "cl100k_tokens", "bert_tokens", "ara_tokens"]].mean()b

In [ ]:
sample["gpt4o_cost"] = sample["gpt4o_tokens"] / 1000 * 0.0001
sample["cl100k_cost"] = sample["cl100k_tokens"] / 1000 * 0.0001
sample["bert_cost"] = sample["bert_tokens"] / 1000 * 0.0001
sample["ara_cost"] = sample["ara_tokens"] / 1000 * 0.0001


In [ ]:
sample.groupby("lang")[["gpt4o_cost", "cl100k_cost", "bert_cost", "ara_cost"]].mean()


In [ ]:
import requests

def generate(prompt, temperature):
    resp = requests.post("http://localhost:11434/api/generate",
        json={"model": "deepseek-r1:1.5b", "prompt": prompt, "temperature": temperature, "stream": False})
    return resp.json()["response"]


In [ ]:
prompt = "Describe a luxury 3-bedroom apartment in Dubai Marina with sea view"
print(generate(prompt, temperature=0))


In [ ]:
for temp in [0, 0.3, 0.8, 1.2]:
    print(f"\n{'='*50}")
    print(f"Temperature: {temp}")
    print(f"{'='*50}")
    print(generate(prompt, temperature=temp))


In [12]:
import requests, time

def ttft(prompt):
    start = time.time()
    resp = requests.post("http://localhost:11434/api/generate",
        json={"model": "deepseek-r1:1.5b", "prompt": prompt, "stream": True},
        stream=True)
    for chunk in resp.iter_lines():
        first_token_time = time.time() - start
        break
    return first_token_time


In [13]:
short = "Describe a luxury apartment. " * 10
medium = "Describe a luxury apartment. " * 100
long = "Describe a luxury apartment. " * 500


In [15]:
print(f"Short: {ttft(short):.3f}s")
print(f"Medium: {ttft(medium):.3f}s")
print(f"Long: {ttft(long):.3f}s")


Short: 1.521s
Medium: 12.236s
Long: 80.852s
